In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Implementing Maps as Lists of Key-Value-Pairs 

The class `ListNode<T>` defines the structure of a single node in the <em style="color:blue">linked lists</em>.
Each node contains three properties:
- `mKey`     stores the *key*,
- `mValue`   stores the *value* associated with this key, and
- `mNextPtr` stores a reference to the next node in the list.  If there is no next node, then 
  `mNextPtr` is `null`.

The class therefore models the `data container` for a single key–value pair.
Its constructor initializes these three properties and sets up the link structure.

In addition to the class definition, an interface with the same name `ListNode<T>` is declared.
This is a specific feature of TypeScript known as interface merging.

While the class defines the runtime structure, the interface extends the type definition by declaring the available methods
`find`, `insert`, `delete`, and `toString`.

This approach allows the later addition of methods using `ListNode.prototype.methodName` in separate cells.

In [ ]:
class ListNode<T> {
  mKey: number;
  mValue: T;
  mNextPtr: ListNode<T> | null;

  constructor(key: number, value: T) {
    this.mKey = key;
    this.mValue = value;
    this.mNextPtr = null;
  }
}
interface ListNode<T> {
  mKey: number;
  mValue: T;
  mNextPtr: ListNode<T> | null;
  find(this: ListNode<T>, key: number): T | null;
  insert(this: ListNode<T>, key: number, value: T): boolean;
  delete(this: ListNode<T>, key: number): any;
  toString(this: ListNode<T>): string;
}

The `find()` method traverses the linked list starting from the current node.  
If a node with the specified key is found, its associated value is returned.  
If no such key exists, the method returns `null`.

In [ ]:
ListNode.prototype.find = function <T>(this: ListNode<T>, key: number): T | null {
  let ptr: ListNode<T> | null = this;
  while (true) {
    if (ptr!.mKey === key) return ptr!.mValue;
    if (ptr!.mNextPtr !== null) {
      ptr = ptr!.mNextPtr;
    } else {
      return null;
    }
  }
};

The `insert()` method adds a new key–value pair `(key, value)` to the list.  
If a node with the same key already exists, its value is updated instead of creating a new node.

The method returns a boolean:
- `true` if a new node was created,  
- `false` if an existing node was updated.

This makes it possible to determine whether the list has been extended or only modified.

In [ ]:
ListNode.prototype.insert = function <T>(
  this: ListNode<T>,
  key: number,
  value: T
): boolean {
  let ptr: ListNode<T> = this;
  while (true) {
    if (ptr.mKey === key) {
      ptr.mValue = value;
      return false;
    } else if (ptr.mNextPtr !== null) {
      ptr = ptr.mNextPtr;
    } else {
      ptr.mNextPtr = new ListNode(key, value);
      return true;
    }
  }
};

The `delete()` method removes the first node containing the specified key.  
If the key is not found, the list remains unchanged.

The method returns an object containing two components:

- `head`: a reference to the (possibly new) head of the list, or `null` if the list becomes empty,  
- `flag`: a boolean indicating whether a deletion occurred.

In [ ]:
ListNode.prototype.delete = function <T>(
  this: ListNode<T>,
  key: number
): { head: ListNode<T> | null; flag: boolean } {
  let previous: ListNode<T> | null = null;
  let ptr: ListNode<T> | null = this;

  while (ptr !== null) {
    if (ptr.mKey === key) {
      if (previous === null) {
        return { head: ptr.mNextPtr, flag: true };
      } else {
        previous.mNextPtr = ptr.mNextPtr;
        return { head: this, flag: true };
      }
    } else if (ptr.mNextPtr !== null) {
      previous = ptr;
      ptr = ptr.mNextPtr;
    } else {
      return { head: this, flag: false };
    }
  }
  return { head: this, flag: false };
};

The `toString()` method returns a string representation of the linked list,  
listing all key–value pairs in order of their appearance.

In [ ]:
ListNode.prototype.toString = function <T>(this: ListNode<T>): string {
  if (this.mNextPtr !== null) {
    return `${this.mKey}: ${this.mValue}, ` + this.mNextPtr.toString();
  } else {
    return `${this.mKey}: ${this.mValue}`;
  }
};

The class `MapIterator<T>` implements the TypeScript interfaces `Iterator<[number, T]>` and `Iterable<[number, T]>`.  

It enables sequential traversal over the elements of the linked list. An internal pointer `mPtr` references the current node and advances with each iteration step.

By implementing the `[Symbol.iterator]()` method, the class becomes compatible with native TypeScript constructs such as `for...of` loops and the spread operator (`[...S]`).

In [ ]:
class MapIterator<T> implements Iterator<[number, T]>, Iterable<[number, T]> {
  private mPtr: ListNode<T> | null;

  constructor(ptr: ListNode<T> | null) {
    this.mPtr = ptr;
  }

  [Symbol.iterator](): IterableIterator<[number, T]> {
    return this;
  }

  next(): IteratorResult<[number, T]> {
    if (this.mPtr === null) {
      return { done: true, value: undefined as any };
    }
    const key = this.mPtr.mKey;
    const value = this.mPtr.mValue;
    this.mPtr = this.mPtr.mNextPtr;
    return { done: false, value: [key, value] };
  }
}

The class `ListMap<T>` represents a **map** implemented as a linked list of key–value pairs.  
It serves as a wrapper for the class `ListNode<T>` and provides a higher-level API for managing entries.

In [ ]:
class ListMap<T> implements Iterable<[number, T]> {
  private mPtr: ListNode<T> | null;

  constructor() {
    this.mPtr = null;
  }

  find(key: number): T | null {
    if (this.mPtr !== null) {
      return this.mPtr.find(key);
    }
    return null;
  }

  insert(key: number, value: T): boolean {
    if (this.mPtr !== null) {
      return this.mPtr.insert(key, value);
    } else {
      this.mPtr = new ListNode(key, value);
      return true;
    }
  }

  delete(key: number): boolean {
    if (this.mPtr !== null) {
      const { head, flag } = this.mPtr.delete(key);
      this.mPtr = head;
      return flag;
    }
    return false;
  }

  [Symbol.iterator](): IterableIterator<[number, T]> {
    return new MapIterator(this.mPtr);
  }

  toString(): string {
    if (this.mPtr !== null) {
      return "{ " + this.mPtr.toString() + " }";
    } else {
      return "{}";
    }
  }
}


The function `main(n = 100)` demonstrates how the `ListMap` class can be used in practice. It applies the **Sieve of Eratosthenes** algorithm to compute all prime numbers up to `n`.

1. All integers from `2` to `n` are inserted into the map.  
2. Multiples of each integer are deleted from the map.  
3. The remaining keys correspond to the prime numbers.

In [ ]:
function main(n = 100): void {
  const S = new ListMap<boolean>();

  for (let i = 2; i <= n; i++) {
    S.insert(i, true);
  }

  for (let i = 2; i <= Math.floor(n / 2) + 1; i++) {
    for (let j = i; j <= Math.floor(n / i) + 1; j++) {
      S.delete(i * j);
    }
  }

  console.log([...S].map(([p, _]) => p));
  console.log(S.find(83));
  console.log(S.find(99));
}

In [ ]:
main();